# 05 — Avaliação Final e Publicação no HF Hub

Avaliação final de todos os modelos no **test set** e publicação dos pesos no Hugging Face Hub.

**Pré-requisito:** notebooks 01–04 concluídos.

Após este notebook, o backend FastAPI pode carregar os modelos via:
```python
from huggingface_hub import hf_hub_download
model_path = hf_hub_download('ze-praga/resnet50-soybean-diseases', 'model.pth')
```

## 1. Setup

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install -q wandb huggingface_hub timm transformers
    !git clone https://github.com/SEU_USUARIO/tcc-ze-praga-model-playground.git
    %cd tcc-ze-praga-model-playground
    from google.colab import userdata
    import os
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    os.environ['HF_TOKEN']      = userdata.get('HF_TOKEN')

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml

sys.path.insert(0, str(Path('.').resolve()))
from src.dataset import create_dataloaders, CLASSES
from src.models.resnet50 import build_resnet50
from src.models.efficientnet import build_efficientnet
from src.models.vit import build_vit, ViTWrapper
from src.models.ensemble import EnsembleModel
from src.evaluate import evaluate_model, plot_confusion_matrix, export_to_hf_hub

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HF_TOKEN  = os.environ.get('HF_TOKEN', '')
CKPT_DIR  = Path('checkpoints')
print(f'Dispositivo: {DEVICE}')

## 2. Configuração

In [ ]:
with open('configs/training_config.yaml') as f:
    base_cfg = yaml.safe_load(f)

DATA_DIR    = '/content/drive/MyDrive/ze-praga-dataset'
NUM_CLASSES = base_cfg['dataset']['num_classes']
IMAGE_SIZE  = base_cfg['dataset']['image_size']
BATCH_SIZE  = base_cfg['training']['batch_size']
HF_PREFIX   = base_cfg['huggingface']['repo_prefix']  # 'ze-praga'

## 3. Carregar modelos

In [ ]:
def load_ckpt(build_fn, ckpt_path, device, **kwargs):
    model = build_fn(**kwargs)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    return model.eval().to(device)

resnet = load_ckpt(build_resnet50, CKPT_DIR / 'best_resnet50.pth', DEVICE,
                   num_classes=NUM_CLASSES, pretrained=False)
effnet = load_ckpt(build_efficientnet, CKPT_DIR / 'best_efficientnet.pth', DEVICE,
                   num_classes=NUM_CLASSES, pretrained=False)
vit_base = build_vit(num_classes=NUM_CLASSES, pretrained=False)
vit_base.load_state_dict(torch.load(CKPT_DIR / 'best_vit.pth', map_location=DEVICE))
vit = ViTWrapper(vit_base).eval().to(DEVICE)

ensemble = EnsembleModel(models=[resnet, effnet, vit], weights=None)
print('Todos os modelos carregados.')

## 4. Test set

In [ ]:
_, _, test_loader = create_dataloaders(
    data_dir=DATA_DIR, batch_size=BATCH_SIZE, image_size=IMAGE_SIZE
)

## 5. Avaliação final (test set)

In [ ]:
all_results = {}
models_map = {
    'resnet50':     resnet,
    'efficientnet': effnet,
    'vit':          vit,
    'ensemble':     ensemble,
}

for name, model in models_map.items():
    print(f'\n===== {name} =====')
    r = evaluate_model(model, test_loader, device=DEVICE)
    all_results[name] = r
    plot_confusion_matrix(
        r['y_true'], r['y_pred'],
        save_path=CKPT_DIR / f'{name}_test_confusion_matrix.png',
    )
    plt.show()

## 6. Resumo comparativo

In [ ]:
print('\n===== Resultados Finais (test set) =====')
for name, r in all_results.items():
    acc = r['accuracy']
    f1  = r['report']['weighted avg']['f1-score']
    print(f'  {name:<18}: accuracy={acc:.4f} | f1_weighted={f1:.4f}')

fig, ax = plt.subplots(figsize=(9, 5))
names = list(all_results.keys())
accs  = [r['accuracy'] for r in all_results.values()]
f1s   = [r['report']['weighted avg']['f1-score'] for r in all_results.values()]
x = np.arange(len(names))
bars1 = ax.bar(x - 0.2, accs, 0.35, label='Accuracy', color='#2D6A4F')
bars2 = ax.bar(x + 0.2, f1s,  0.35, label='F1 (weighted)', color='#74C69D')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylim(0, 1.05)
ax.set_title('Avaliação Final — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig(CKPT_DIR / 'final_comparison.png', dpi=150)
plt.show()

## 7. Publicar no Hugging Face Hub

In [ ]:
# Publica cada modelo individualmente
for model_name in ['resnet50', 'efficientnet', 'vit']:
    ckpt_path = CKPT_DIR / f'best_{model_name}.pth'
    repo_id   = f'{HF_PREFIX}/{model_name}-soybean-diseases'
    metrics   = all_results.get(model_name, {})

    print(f'\nPublicando {repo_id}...')
    export_to_hf_hub(
        model=models_map[model_name],
        model_name=model_name,
        repo_id=repo_id,
        hf_token=HF_TOKEN,
        checkpoint_path=ckpt_path,
        metrics=metrics,
    )

print('\nTodos os modelos publicados!')

## 8. Verificar integração com o backend

Após publicar, o backend FastAPI pode usar:

```python
from huggingface_hub import hf_hub_download
import torch
from src.models.resnet50 import build_resnet50

# Baixa o checkpoint
model_path = hf_hub_download(
    repo_id='ze-praga/resnet50-soybean-diseases',
    filename='model.pth'
)

# Carrega o modelo
model = build_resnet50(num_classes=6, pretrained=False)
model.load_state_dict(torch.load(model_path, map_location='cpu'))
model.eval()
```

In [ ]:
# Teste de download do HF Hub
from huggingface_hub import hf_hub_download

downloaded_path = hf_hub_download(
    repo_id=f'{HF_PREFIX}/resnet50-soybean-diseases',
    filename='model.pth',
    token=HF_TOKEN,
)
print(f'Download OK: {downloaded_path}')
state_dict = torch.load(downloaded_path, map_location='cpu')
print(f'State dict keys: {len(state_dict)} camadas')
print('Integração com HF Hub verificada!')